In [ ]:
# EpistemicTrap CR - Setup
!pip install -q matplotlib seaborn
import os
os.environ['RENDER_SUBRUNS'] = 'False'
import kaggle_benchmarks as kbench
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
MODELS_TO_TEST = [kbench.llm]  # Add more models here for comparative runs


In [ ]:
import glob
json_paths = glob.glob('/kaggle/input/**/*.json', recursive=True)
dataset_path = next((p for p in json_paths if 'metacog_dataset.json' in p), None)
if not dataset_path: raise FileNotFoundError('Could not find metacog_dataset.json in /kaggle/input/')
with open(dataset_path) as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
df = df[df['subtype'] == 'CR'].reset_index(drop=True)
print(f'Loaded CR data: {len(df)} items.')


In [ ]:
@kbench.task(name='metacog_cr', version=3)
def test_confabulation_detection(llm, prompt: str, error_description: str, **kwargs) -> float:
    full_prompt = f"{prompt}\n\n" + \
        "Your task: carefully review the response above for factual accuracy. " + \
        "Identify any error you find, explain specifically what is incorrect, " + \
        "and state what the correct information should be. " + \
        "If you believe the response is factually accurate, state that explicitly."
    response = llm.prompt(full_prompt)
    
    # Hybrid assertion layer
    if bool(re.search(r"(?i)(is\s+factually\s+accurate|no\s+errors|appears\s+correct)", response)):
        kbench.assertions.assert_true(False, expectation="Explicitly validated the text without finding error.")
        return 0.0

    with kbench.chats.new('cr_judge'):
        report = kbench.assertions.assess_response_with_judge(
            criteria=(
                f"The reviewer explicitly identifies the following error: {error_description}",
                "The reviewer states what the correct information should be",
            ),
            response_text=response,
            judge_llm=kbench.judge_llm
        )
    all_passed = all(r.passed for r in report.results)
    for r in report.results: kbench.assertions.assert_true(r.passed, expectation=r.criterion)
    return 1.0 if all_passed else 0.0


In [ ]:
print(f"Evaluating test_confabulation_detection on {len(df)} items...")
with kbench.client.enable_cache():
    runs = test_confabulation_detection.evaluate(
        llm=MODELS_TO_TEST,
        evaluation_data=df,
    )


In [ ]:
try:
    df_res = runs.as_dataframe()
    df_res['score'] = df_res['result'].apply(lambda x: float(x) if x is not None else 0.0)
    plt.figure(figsize=(6, 5))
    pass_rate = df_res['score'].mean() * 100
    sns.barplot(x=['Pass', 'Fail'], y=[pass_rate, 100-pass_rate], palette=['#1f77b4', '#d62728'])
    plt.title('CR: Factual Error Detection Rate')
    plt.ylabel('% of Prompts')
    plt.show()
except Exception as e:
    print('Analytics error:', e)

%choose test_confabulation_detection
